# Fetch list of all ports for a given bounding box

In [ ]:
from aisdb.ports.api import WorldPortIndexClient # for weather

# Initialize the client
client = WorldPortIndexClient()

# Query Gulf of St. Lawrence region
# df_ports = client.fetch_ports(ymin, ymax, xmin, xmax, out_path="gulf_ports.csv") # save out to a file
df_ports = client.fetch_ports(
    lat_min=45.0,
    lat_max=51.5,
    lon_min=-71.5,
    lon_max=-55.0
)
# Filter for cargo-capable ports
df_cargo = client.filter_by_cargo_depth(df_ports)

In [ ]:
df_ports.head(5)

# Get H3 indexes for the ports

In [ ]:
from aisdb.discretize.h3 import Discretizer

# Initialize discretizer
discretizer = Discretizer(resolution=6)

# Rename and cast lat/lon columns
df_ports = df_ports.rename(columns={"LAT": "lat", "LON": "lon"})
df_ports[["lat", "lon"]] = df_ports[["lat", "lon"]].astype(float)

# Apply get_h3_index row-wise
df_ports["h3_index_res_6"] = df_ports.apply(lambda row: discretizer.get_h3_index(row["lat"], row["lon"]), axis=1)

In [ ]:
print(df_ports[["PORT_NAME","lat", "lon", "h3_index_res_6","HARBORSIZE"]].head(10))

In [ ]:
import plotly.express as px

# Ensure port_name exists
df_ports = df_ports.rename(columns={"PORT_NAME": "port_name"}) if "PORT_NAME" in df_ports.columns else df_ports.assign(port_name="Port")

# Add sailing anchor emoji to hover
df_ports["hover_label"] = df_ports["port_name"].apply(lambda x: f"⚓ {x}")

# Assign each port a unique color
fig = px.scatter_map(
    df_ports,
    lat="lat",
    lon="lon",
    hover_name="hover_label",
    color="port_name",  # different color for each port
    zoom=5,
    height=700
)

fig.update_layout(
    mapbox_style="open-street-map",  # or "stamen-terrain", "carto-positron", etc.
    margin={"r": 0, "t": 0, "l": 0, "b": 0},
    legend_title_text="⚓ Ports"
)

fig.show()
